# Régressions du salaire selon le sexe, la CSP et l’âge

On étudie les associations entre le salaire net moyen mensuel, le sexe, la catégorie socio-professionnelle (CSP) et l’âge. L’analyse porte sur les données du secteur privé, en euros constants.

Les modèles incluent des effets fixes par année pour tenir compte de l’évolution générale des salaires. Les données ne croisent pas l’âge et la CSP : les observations détaillées par âge ont la CSP « Ensemble », tandis que les observations détaillées par CSP ont l’âge « Ensemble ». On estime donc deux modèles complémentaires, et non un modèle contrôlant simultanément l’âge et la CSP. Les observations sont des **moyennes agrégées par groupe** : les résultats décrivent des associations, pas des effets causaux individuels.

## 1. Chargement et préparation des données

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


def formater_p_value(valeur):
    if valeur < 0.001:
        return '< 0,001'
    return f'{valeur:.3f}'.replace('.', ',')


def trouver_racine_projet():
    courant = Path.cwd().resolve()
    for dossier in (courant, *courant.parents):
        if (dossier / 'Data nettoyees').is_dir():
            return dossier
    raise FileNotFoundError(
        'Dossier « Data nettoyees » introuvable. '
        'Lancez ce notebook depuis le projet ou un de ses sous-dossiers.'
    )


RACINE = trouver_racine_projet()
CHEMIN_DONNEES = (
    RACINE
    / 'Data nettoyees'
    / 'DS_DERA_PRIVE_SERIES_LONGUES_data_nettoye.csv'
)
if not CHEMIN_DONNEES.is_file():
    raise FileNotFoundError(
        f'Fichier de données nettoyées absent : {CHEMIN_DONNEES}. '
        'Lancez d’abord Nettoyage_donnees.ipynb.'
    )

prive = pd.read_csv(CHEMIN_DONNEES, sep=';', encoding='utf-8-sig')
print(f'Fichier chargé : {CHEMIN_DONNEES.name} ({len(prive):,} lignes)')
print(f'Colonnes disponibles : {", ".join(prive.columns)}')

## 2. Construction de l’échantillon

On conserve le salaire net moyen mensuel, pour la France et l’ensemble des secteurs, âges, tailles d’entreprise et temps de travail. On écarte les modalités « Ensemble » du sexe et de la CSP afin de comparer les femmes et les hommes dans chacune des CSP.

In [ ]:
SALAIRE = 'Salaire net moyen mensuel (Euros constants)'
colonnes_requises = {
    'indicateur', 'centile_salaire', 'geo', 'secteur_activite',
    'tranche_age', 'taille_entreprise', 'temps_travail',
    'categorie_socio_prof', 'sexe', 'annee', 'valeur',
}
colonnes_absentes = colonnes_requises.difference(prive.columns)
if colonnes_absentes:
    raise ValueError(f'Colonnes manquantes dans le fichier : {sorted(colonnes_absentes)}')

echantillon = prive.loc[
    prive['indicateur'].eq(SALAIRE)
    & prive['centile_salaire'].eq('Ensemble')
    & prive['geo'].eq('F')
    & prive['secteur_activite'].eq('Ensemble des secteurs')
    & prive['tranche_age'].eq('Ensemble')
    & prive['taille_entreprise'].eq('Ensemble')
    & prive['temps_travail'].eq('Ensemble')
    & prive['sexe'].isin(['Femme', 'Homme'])
    & prive['categorie_socio_prof'].ne('Ensemble'),
    ['annee', 'sexe', 'categorie_socio_prof', 'valeur'],
].copy()

echantillon['annee'] = pd.to_numeric(echantillon['annee'], errors='coerce')
echantillon['valeur'] = pd.to_numeric(echantillon['valeur'], errors='coerce')
if echantillon[['annee', 'valeur']].isna().any().any():
    raise ValueError('Des années ou des salaires sont manquants ou non numériques dans l’échantillon.')
echantillon['annee'] = echantillon['annee'].astype(int)
echantillon['femme'] = echantillon['sexe'].eq('Femme').astype(int)

cles = ['annee', 'sexe', 'categorie_socio_prof']
if echantillon.duplicated(cles).any():
    raise ValueError(f'Plusieurs observations pour les mêmes clés : {cles}')

annees = sorted(echantillon['annee'].unique())
sexes = ['Femme', 'Homme']
csp = sorted(echantillon['categorie_socio_prof'].unique())
attendues = pd.MultiIndex.from_product(
    [annees, sexes, csp], names=cles
)
observees = pd.MultiIndex.from_frame(echantillon[cles])
manquantes = attendues.difference(observees)
if len(manquantes):
    raise ValueError(f'Panel incomplet : {len(manquantes)} combinaisons année-sexe-CSP manquent.')

print(
    f"Échantillon : {len(echantillon)} observations, {len(annees)} années "
    f"({min(annees)}–{max(annees)}), {len(csp)} CSP et 2 sexes."
)
print(f'CSP observées : {", ".join(csp)}')
display(echantillon.head())

## 3. Quelques repères descriptifs

Le tableau et le graphique servent à situer les données avant la régression. Chaque série du graphique correspond à une CSP et un sexe.

In [ ]:
resume = (
    echantillon.groupby(['categorie_socio_prof', 'sexe'])['valeur']
    .agg(['count', 'mean', 'std', 'min', 'max'])
    .rename(columns={
        'count': 'Nombre d’années',
        'mean': 'Salaire moyen (€ constants)',
        'std': 'Écart-type entre années',
        'min': 'Minimum',
        'max': 'Maximum',
    })
)
display(resume.round(1))

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for categorie, ax in zip(csp, axes.flat):
    donnees_csp = echantillon[echantillon['categorie_socio_prof'].eq(categorie)]
    for sexe in sexes:
        serie = donnees_csp[donnees_csp['sexe'].eq(sexe)].sort_values('annee')
        ax.plot(serie['annee'], serie['valeur'], marker='.', label=sexe)
    ax.set_title(categorie)
    ax.set_ylabel('Salaire mensuel (€ constants)')
    ax.grid(alpha=0.3)
    ax.legend()
for ax in axes[-1, :]:
    ax.set_xlabel('Année')
fig.suptitle('Salaire net moyen selon le sexe et la CSP')
fig.tight_layout()
plt.show()

## 4. Régression linéaire avec effets fixes d’année

Le modèle estimé est :

$$Salaire_{t,s,c} = \alpha + \beta_F Femme_s + \beta_c CSP_c + \delta_c(Femme_s \times CSP_c) + \lambda_t + \varepsilon_{t,s,c}$$

La CSP de référence est **Ouvriers** et le sexe de référence est **Homme**. Les erreurs-types sont regroupées par année, car les observations d’une même année partagent des conditions économiques communes. Les effets fixes d’année absorbent les différences de niveau propres à chaque année.

Pour chaque coefficient, le test de significativité vérifie l’hypothèse nulle $H_0 : \beta=0$. Une p-value inférieure à 0,05 est considérée comme statistiquement significative au seuil de 5 %. Une p-value plus élevée signifie qu’on ne rejette pas cette hypothèse au seuil retenu ; ce n’est pas une preuve que l’effet est exactement nul. Les tests sont approximatifs et doivent être interprétés prudemment, notamment parce qu’il y a 29 années regroupées.

In [ ]:
formule = (
    "valeur ~ femme * C(categorie_socio_prof, Treatment(reference='Ouvriers')) "
    "+ C(annee)"
)
modele = smf.ols(formula=formule, data=echantillon).fit(
    cov_type='cluster',
    cov_kwds={'groups': echantillon['annee'], 'use_correction': True},
)
print(f'Formule : {formule}')
print(f"Observations : {int(modele.nobs)} ; années regroupées : {echantillon['annee'].nunique()}")
print(f"R² : {modele.rsquared:.3f} ; R² ajusté : {modele.rsquared_adj:.3f}")

## 5. Résultats et interprétation des coefficients

Le tableau ci-dessous met en avant les coefficients correspondant au sexe, à la CSP et à leur interaction. Les indicatrices d’année sont omises de ce tableau synthétique, mais restent incluses dans l’estimation.

In [ ]:
termes = modele.params.index
masque_interet = (
    (termes == 'femme')
    | termes.str.startswith('C(categorie_socio_prof')
    | termes.str.startswith('femme:C(categorie_socio_prof')
)
intervalle = modele.conf_int()
table_coefficients = pd.DataFrame({
    'Coefficient (€)': modele.params[masque_interet],
    'IC 95 % bas (€)': intervalle.loc[masque_interet, 0],
    'IC 95 % haut (€)': intervalle.loc[masque_interet, 1],
    'Erreur-type cluster année (€)': modele.bse[masque_interet],
    'p-value': modele.pvalues[masque_interet],
})
table_coefficients['Significatif à 5 %'] = np.where(
    table_coefficients['p-value'] < 0.05, 'Oui', 'Non'
)
table_coefficients_affichage = table_coefficients.copy()
table_coefficients_affichage['p-value'] = table_coefficients_affichage['p-value'].map(formater_p_value)
display(table_coefficients_affichage.round(3))

print('Interprétation des coefficients (en euros mensuels constants) :')
for terme, ligne in table_coefficients.iterrows():
    coefficient = ligne['Coefficient (€)']
    p_value = ligne['p-value']
    decision = (
        'association statistiquement significative au seuil de 5 %'
        if p_value < 0.05
        else 'association non significative au seuil de 5 %'
    )
    if terme == 'femme':
        lecture = 'écart Femme–Homme chez les ouvriers'
    elif terme.startswith('femme:C(categorie_socio_prof'):
        categorie = terme.rsplit('[T.', 1)[-1].rstrip(']')
        lecture = f'différence de l’écart Femme–Homme entre {categorie} et les ouvriers'
    else:
        categorie = terme.rsplit('[T.', 1)[-1].rstrip(']')
        lecture = f'écart de salaire entre {categorie} et les ouvriers chez les hommes'
    print(f'- {lecture} : {coefficient:+.0f} € ; {decision} (p {formater_p_value(p_value)}).')

def test_conjoint(resultat, termes_testes):
    if not termes_testes:
        raise ValueError('La liste des coefficients à tester ne peut pas être vide.')
    matrice = np.zeros((len(termes_testes), len(resultat.params)))
    for ligne, terme in enumerate(termes_testes):
        matrice[ligne, resultat.params.index.get_loc(terme)] = 1
    return resultat.wald_test(matrice, use_f=True, scalar=True)

termes_csp = [
    terme for terme in modele.params.index
    if terme.startswith('C(categorie_socio_prof')
]
test_csp = test_conjoint(modele, termes_csp)
p_csp = float(np.asarray(test_csp.pvalue).reshape(-1)[0])
print(
    '\nTest conjoint des CSP chez les hommes '
    '(H₀ : les écarts avec les ouvriers sont tous nuls) : '
    f"F = {float(np.asarray(test_csp.statistic).reshape(-1)[0]):.2f}, p {formater_p_value(p_csp)}. "
    + ('Au moins une CSP diffère significativement des ouvriers au seuil de 5 %.'
       if p_csp < 0.05 else
       'On ne détecte pas de différence globale significative entre CSP au seuil de 5 %.')
)

interactions = [
    terme for terme in modele.params.index
    if terme.startswith('femme:C(categorie_socio_prof')
]
if not interactions:
    raise ValueError('Aucun coefficient d’interaction sexe-CSP trouvé.')
test_interactions = np.zeros((len(interactions), len(modele.params)))
for ligne, terme in enumerate(interactions):
    test_interactions[ligne, modele.params.index.get_loc(terme)] = 1
test_global_interactions = modele.wald_test(
    test_interactions, use_f=True, scalar=True
)
p_global = float(np.asarray(test_global_interactions.pvalue).reshape(-1)[0])
print(
    '\nTest conjoint des interactions (H₀ : tous les écarts supplémentaires '
    f'par CSP sont nuls) : F = {float(np.asarray(test_global_interactions.statistic).reshape(-1)[0]):.2f}, '
    f"p {formater_p_value(p_global)}. "
    + ('Les écarts femmes-hommes varient significativement selon la CSP au seuil de 5 %.'
       if p_global < 0.05 else
       'On ne détecte pas de variation significative des écarts femmes-hommes selon la CSP au seuil de 5 %.')
)

## 6. Écart Femme–Homme estimé dans chaque CSP

Pour chaque CSP, on calcule le contraste linéaire correspondant à l’écart Femme–Homme. Les intervalles de confiance utilisent toute la matrice de covariance (y compris les covariances entre coefficients), plutôt qu’une simple addition des erreurs-types.

In [ ]:
nom_csp = 'C(categorie_socio_prof, Treatment(reference='
nom_effet_femme = 'femme'
ecarts = []
for categorie in csp:
    contraste = np.zeros(len(modele.params))
    contraste[modele.params.index.get_loc(nom_effet_femme)] = 1
    if categorie != 'Ouvriers':
        nom_interaction = (
            f"femme:{nom_csp}'Ouvriers'))[T.{categorie}]"
        )
        if nom_interaction not in modele.params.index:
            raise KeyError(f'Coefficient d’interaction introuvable : {nom_interaction}')
        contraste[modele.params.index.get_loc(nom_interaction)] = 1

    test = modele.t_test(contraste)
    borne_inf, borne_sup = np.asarray(test.conf_int()).reshape(-1)
    ecarts.append({
        'CSP': categorie,
        'Écart Femme–Homme (€)': float(np.asarray(test.effect).reshape(-1)[0]),
        'Borne inférieure 95 %': borne_inf,
        'Borne supérieure 95 %': borne_sup,
        'p-value': float(np.asarray(test.pvalue).reshape(-1)[0]),
    })

table_ecarts = pd.DataFrame(ecarts).set_index('CSP')
table_ecarts['Significatif à 5 %'] = np.where(
    table_ecarts['p-value'] < 0.05, 'Oui', 'Non'
)
table_ecarts_affichage = table_ecarts.copy()
table_ecarts_affichage['p-value'] = table_ecarts_affichage['p-value'].map(formater_p_value)
display(table_ecarts_affichage.round(2))
for categorie, ligne in table_ecarts.iterrows():
    direction = 'inférieur' if ligne['Écart Femme–Homme (€)'] < 0 else 'supérieur'
    if ligne['p-value'] < 0.05:
        print(
            f"- {categorie} : le salaire moyen estimé des femmes est "
            f"{direction} de {abs(ligne['Écart Femme–Homme (€)']):.0f} € à celui des hommes "
            f"(IC 95 % [{ligne['Borne inférieure 95 %']:.0f} ; "
            f"{ligne['Borne supérieure 95 %']:.0f}] € ; p {formater_p_value(ligne['p-value'])})."
        )
    else:
        print(
            f"- {categorie} : écart estimé de {ligne['Écart Femme–Homme (€)']:+.0f} €, "
            f"mais non significatif au seuil de 5 % (p {formater_p_value(ligne['p-value'])}) ; "
            'on ne peut pas conclure à un écart statistiquement différent de zéro.'
        )

## 7. Bilan du modèle sexe-CSP et limites

- Le coefficient `femme` donne l’écart moyen Femme–Homme pour la CSP de référence (ouvriers), à année donnée.
- Les interactions indiquent comment cet écart varie selon les autres CSP ; les contrastes précédents donnent l’écart total estimé pour chaque CSP.
- Les salaires sont des moyennes agrégées : on ne contrôle pas ici les caractéristiques individuelles (ancienneté, métier précis, durée de travail individuelle, etc.).
- Une p-value significative indique que l’écart estimé est difficilement compatible avec un coefficient nul selon le modèle ; elle ne mesure ni l’importance pratique ni la probabilité que l’hypothèse nulle soit vraie.
- Le regroupement des erreurs-types par année tient compte d’une dépendance possible entre observations de la même année, mais le nombre de groupes (29 années) reste modeste.
- L’âge et la CSP ne sont pas observés simultanément dans les mêmes lignes détaillées. Il faut donc lire séparément le modèle sexe-CSP et le modèle âge-sexe ci-dessous.
- Ces résultats agrégés sont descriptifs et associatifs. Ils ne constituent pas une estimation causale d’une pénalité salariale individuelle ni une preuve isolée de discrimination.

## 8. Modèle complémentaire : âge et sexe

Les données disponibles pour étudier l’âge détaillent six tranches d’âge, mais indiquent la CSP « Ensemble ». Ce modèle estime donc les écarts liés à l’âge et au sexe **sans contrôler la CSP**. La tranche **31 à 40 ans** sert de référence. Les interactions âge-sexe permettent de vérifier si l’écart femmes-hommes varie selon l’âge.

In [ ]:
echantillon_age = prive.loc[
    prive['indicateur'].eq(SALAIRE)
    & prive['centile_salaire'].eq('Ensemble')
    & prive['geo'].eq('F')
    & prive['secteur_activite'].eq('Ensemble des secteurs')
    & prive['tranche_age'].ne('Ensemble')
    & prive['taille_entreprise'].eq('Ensemble')
    & prive['temps_travail'].eq('Ensemble')
    & prive['sexe'].isin(['Femme', 'Homme'])
    & prive['categorie_socio_prof'].eq('Ensemble'),
    ['annee', 'sexe', 'tranche_age', 'valeur'],
].copy()
echantillon_age['annee'] = pd.to_numeric(echantillon_age['annee'], errors='coerce')
echantillon_age['valeur'] = pd.to_numeric(echantillon_age['valeur'], errors='coerce')
if echantillon_age[['annee', 'valeur']].isna().any().any():
    raise ValueError('Des années ou des salaires manquent dans les données détaillées par âge.')
echantillon_age['annee'] = echantillon_age['annee'].astype(int)
echantillon_age['femme'] = echantillon_age['sexe'].eq('Femme').astype(int)

ages = sorted(echantillon_age['tranche_age'].unique())
annees_age = sorted(echantillon_age['annee'].unique())
cles_age = ['annee', 'sexe', 'tranche_age']
if echantillon_age.duplicated(cles_age).any():
    raise ValueError(f'Plusieurs observations pour les mêmes clés : {cles_age}')
attendues_age = pd.MultiIndex.from_product(
    [annees_age, sexes, ages], names=cles_age
)
observees_age = pd.MultiIndex.from_frame(echantillon_age[cles_age])
manquantes_age = attendues_age.difference(observees_age)
if len(manquantes_age):
    raise ValueError(f'Panel âge-sexe incomplet : {len(manquantes_age)} combinaisons manquent.')
print(
    f"Échantillon âge-sexe : {len(echantillon_age)} observations, "
    f"{len(ages)} tranches d’âge, de {min(annees_age)} à {max(annees_age)}."
)
display(
    echantillon_age.groupby(['tranche_age', 'sexe'])['valeur']
    .agg(['count', 'mean', 'std']).round(1)
)

Le modèle utilise les tranches d’âge comme variables catégorielles plutôt que d’attribuer un âge numérique arbitraire à chaque tranche. Le coefficient `femme` compare femmes et hommes de 31–40 ans ; les autres coefficients d’âge comparent les hommes à ceux de 31–40 ans ; les interactions mesurent les différences entre écarts femmes-hommes.

In [ ]:
formule_age = (
    "valeur ~ femme * C(tranche_age, Treatment(reference='De 31 à 40 ans')) "
    "+ C(annee)"
)
modele_age = smf.ols(formula=formule_age, data=echantillon_age).fit(
    cov_type='cluster',
    cov_kwds={'groups': echantillon_age['annee'], 'use_correction': True},
)
print(f'Formule : {formule_age}')
print(
    f"Observations : {int(modele_age.nobs)} ; années regroupées : "
    f"{echantillon_age['annee'].nunique()} ; R² : {modele_age.rsquared:.3f}."
)

In [ ]:
termes_age = modele_age.params.index
masque_age = (
    (termes_age == 'femme')
    | termes_age.str.startswith('C(tranche_age')
    | termes_age.str.startswith('femme:C(tranche_age')
)
intervalle_age = modele_age.conf_int()
table_age = pd.DataFrame({
    'Coefficient (€)': modele_age.params[masque_age],
    'IC 95 % bas (€)': intervalle_age.loc[masque_age, 0],
    'IC 95 % haut (€)': intervalle_age.loc[masque_age, 1],
    'p-value': modele_age.pvalues[masque_age],
})
table_age['Significatif à 5 %'] = np.where(table_age['p-value'] < 0.05, 'Oui', 'Non')
table_age_affichage = table_age.copy()
table_age_affichage['p-value'] = table_age_affichage['p-value'].map(formater_p_value)
display(table_age_affichage.round(3))
print('Interprétation des coefficients (en euros mensuels constants) :')
for terme, ligne in table_age.iterrows():
    coefficient = ligne['Coefficient (€)']
    decision = (
        'significatif au seuil de 5 %'
        if ligne['p-value'] < 0.05
        else 'non significatif au seuil de 5 %'
    )
    if terme == 'femme':
        lecture = 'écart Femme–Homme chez les 31–40 ans'
    elif terme.startswith('femme:C(tranche_age'):
        age = terme.rsplit('[T.', 1)[-1].rstrip(']')
        lecture = f'différence de l’écart Femme–Homme entre {age} et les 31–40 ans'
    else:
        age = terme.rsplit('[T.', 1)[-1].rstrip(']')
        lecture = f'écart entre les hommes de {age} et ceux de 31–40 ans'
    print(f"- {lecture} : {coefficient:+.0f} € ; {decision} (p {formater_p_value(ligne['p-value'])}).")

termes_age_principaux = [
    terme for terme in modele_age.params.index
    if terme.startswith('C(tranche_age')
]
test_age_principal = test_conjoint(modele_age, termes_age_principaux)
p_age_principal = float(np.asarray(test_age_principal.pvalue).reshape(-1)[0])
print(
    '\nTest conjoint des tranches d’âge chez les hommes '
    '(H₀ : les écarts avec les 31–40 ans sont tous nuls) : '
    f"F = {float(np.asarray(test_age_principal.statistic).reshape(-1)[0]):.2f}, "
    f"p {formater_p_value(p_age_principal)}. "
    + ('Au moins une tranche d’âge diffère significativement des 31–40 ans au seuil de 5 %.'
       if p_age_principal < 0.05 else
       'On ne détecte pas de différence globale significative entre âges au seuil de 5 %.')
)

interactions_age = [
    terme for terme in modele_age.params.index
    if terme.startswith('femme:C(tranche_age')
]
test_interactions_age = test_conjoint(modele_age, interactions_age)
p_interactions_age = float(np.asarray(test_interactions_age.pvalue).reshape(-1)[0])
print(
    '\nTest conjoint des interactions âge-sexe '
    '(H₀ : l’écart femmes-hommes ne varie pas selon l’âge) : '
    f"F = {float(np.asarray(test_interactions_age.statistic).reshape(-1)[0]):.2f}, "
    f"p {formater_p_value(p_interactions_age)}. "
    + ('Les différences d’écarts selon l’âge sont significatives au seuil de 5 %.'
       if p_interactions_age < 0.05 else
       'On ne détecte pas de différence significative des écarts selon l’âge au seuil de 5 %.')
)

ecarts_age = []
for age in ages:
    contraste = np.zeros(len(modele_age.params))
    contraste[modele_age.params.index.get_loc('femme')] = 1
    if age != 'De 31 à 40 ans':
        correspondances = [
            terme for terme in interactions_age if terme.endswith(f'[T.{age}]')
        ]
        if len(correspondances) != 1:
            raise KeyError(f'Interaction âge-sexe introuvable ou ambiguë pour : {age}')
        contraste[modele_age.params.index.get_loc(correspondances[0])] = 1
    test = modele_age.t_test(contraste)
    borne_inf, borne_sup = np.asarray(test.conf_int()).reshape(-1)
    ecarts_age.append({
        'Tranche d’âge': age,
        'Écart Femme–Homme (€)': float(np.asarray(test.effect).reshape(-1)[0]),
        'IC 95 % bas (€)': borne_inf,
        'IC 95 % haut (€)': borne_sup,
        'p-value': float(np.asarray(test.pvalue).reshape(-1)[0]),
    })
table_ecarts_age = pd.DataFrame(ecarts_age).set_index('Tranche d’âge')
table_ecarts_age['Significatif à 5 %'] = np.where(
    table_ecarts_age['p-value'] < 0.05, 'Oui', 'Non'
)
table_ecarts_age_affichage = table_ecarts_age.copy()
table_ecarts_age_affichage['p-value'] = table_ecarts_age_affichage['p-value'].map(formater_p_value)
display(table_ecarts_age_affichage.round(2))
for age, ligne in table_ecarts_age.iterrows():
    decision = (
        'écart statistiquement significatif au seuil de 5 %'
        if ligne['p-value'] < 0.05
        else 'écart non significatif au seuil de 5 %'
    )
    print(
        f"- {age} : écart estimé Femme–Homme de "
        f"{ligne['Écart Femme–Homme (€)']:+.0f} € ({decision}, "
        f"p {formater_p_value(ligne['p-value'])})."
    )

## 9. Conclusion d’ensemble

Les régressions permettent de comparer les salaires moyens selon le sexe et, séparément, selon le sexe et la CSP ou selon le sexe et l’âge. Les p-values et intervalles de confiance quantifient l’incertitude statistique sous les hypothèses du modèle. La structure de la base empêche de distinguer simultanément l’âge et la CSP ; aucun résultat ne doit donc être interprété comme un écart « toutes choses égales par ailleurs » entre individus comparables. L’analyse porte sur des agrégats, ne tient pas compte de caractéristiques telles que l’ancienneté ou le métier précis, et ne suffit pas à démontrer une causalité ou une discrimination individuelle.